# Customer Churn and Retention Analysis
### End-to-End Data Science, Machine Learning, and Strategic Retention Analytics

**Objective:** Identify customers likely to churn, uncover root causes of attrition, quantify financial exposure, and formulate data-driven retention playbooks using Machine Learning, SQL, and Exploratory Data Analysis.

**Dataset:** IBM Telco Customer Churn (7,043 customer accounts, 21 attributes).

In [ ]:
import sys
import os
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_cleaning import load_raw_data, clean_customer_data
from src.exploratory_analysis import calculate_churn_kpis
from src.feature_engineering import CustomerFeatureEngineer, build_preprocessor
from src.model_training import train_and_evaluate_models
from src.customer_segmentation import perform_customer_clustering
from src.retention_recommendations import answer_business_questions, get_company_retention_playbook

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Environment initialized successfully.")

## 1. Data Ingestion & Overview
Loading raw IBM Telco Customer Churn dataset and inspecting dimensions and column schema.

In [ ]:
raw_df = load_raw_data()
print(f"Raw dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head(5)

## 2. Data Cleaning & Preprocessing
- Identification and imputation of blank strings in `TotalCharges` (11 rows with `tenure == 0`).
- Type conversions: numeric casting for `TotalCharges`, `tenure`, and `SeniorCitizen`.
- Duplicate detection and whitespace stripping.
- IQR Outlier diagnostics.

In [ ]:
cleaned_df, cleaning_stats = clean_customer_data(raw_df)
print("Cleaning Summary:")
for k, v in cleaning_stats.items():
    print(f"  - {k}: {v}")
cleaned_df.info()

## 3. Exploratory Data Analysis (EDA) & Business KPIs
Computing high-level customer retention metrics, churn rate, and annual revenue at risk.

In [ ]:
kpis = calculate_churn_kpis(cleaned_df)
kpi_summary = pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])
kpi_summary

In [ ]:
# Visualizing Churn by Contract and Payment Method
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
contract_churn = cleaned_df.groupby("Contract")["Churn_Numeric"].mean() * 100
contract_churn.plot(kind="bar", ax=axes[0], color=["#d95f02", "#7570b3", "#2b5c8f"], edgecolor="black")
axes[0].set_title("Churn Rate by Contract Type (%)", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")

payment_churn = cleaned_df.groupby("PaymentMethod")["Churn_Numeric"].mean() * 100
payment_churn.sort_values().plot(kind="barh", ax=axes[1], color="#2b5c8f", edgecolor="black")
axes[1].set_title("Churn Rate by Payment Method (%)", fontweight="bold")
axes[1].set_xlabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

## 4. Machine Learning Model Training & Benchmarking
Benchmarking 4 classification algorithms using Scikit-Learn Pipelines with Stratified 80/20 train-test splitting:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. Gradient Boosting Classifier

In [ ]:
results, comparison_df, best_pipe, best_meta = train_and_evaluate_models(cleaned_df)
print(f"Best Performing Model: {best_meta['best_model_name']}")
comparison_df

## 5. Customer Behavioral Segmentation (K-Means)
Clustering customers into 4 behavioral archetypes based on tenure, monthly spend, lifetime spend, and service adoption breadth.

In [ ]:
df_segmented, profile_summary, kmeans, scaler = perform_customer_clustering(cleaned_df, k=4)
profile_summary

## 6. Retention Recommendations & Answers to Core Business Questions
Actionable strategic initiatives mapped to root causes identified across the data.

In [ ]:
answers = answer_business_questions(kpis)
for q_id, q_data in answers.items():
    print(f"\n[{q_id}] {q_data['question']}")
    print(f"  {q_data['answer']}")